# Giai đoạn 2: Tiền xử lý & Làm sạch dữ liệu PR (Data Preprocessing & Cleaning)
Dự án: **Phân tích Đối chiếu & Xây dựng Mô hình Cảnh báo Sớm cho Pull Request**
Thành viên thực hiện: **Trần Đức Thịnh (Data Analyst)**

Sổ tay này thực hiện các công việc:
- Đọc dữ liệu từ View SQLite `view_pr_clean`.
- Xử lý giá trị thiếu (Missing values).
- Tính toán lại thời gian sống thực tế của PR (`duration_minutes`) từ lúc mở đến lúc đóng.
- Lọc bỏ các PR ngoại lệ (Outliers) dựa trên phân phối dữ liệu thực tế.
- Lưu tập dữ liệu sạch ra file `data/processed/clean_prs.csv`.

In [1]:
import os
import pandas as pd
import numpy as np

# Tự động phát hiện thư mục gốc của dự án (Project Root)
def find_project_root():
    path = os.getcwd()
    while path and not (os.path.exists(os.path.join(path, "database")) and os.path.exists(os.path.join(path, "data"))):
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return path

BASE_DIR = find_project_root()
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

DB_PATH = os.path.join(BASE_DIR, "database", "github_prs.db")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
OUTPUT_PATH = os.path.join(PROCESSED_DIR, "clean_prs.csv")

print(f"Base Directory: {BASE_DIR}")
print(f"Database Path: {DB_PATH}")
print(f"Output Path: {OUTPUT_PATH}")


Base Directory: e:\_FPT_UNI_\Ki 3\ADY\ADY_Final_Project
Database Path: e:\_FPT_UNI_\Ki 3\ADY\ADY_Final_Project\database\github_prs.db
Output Path: e:\_FPT_UNI_\Ki 3\ADY\ADY_Final_Project\data\processed\clean_prs.csv


### 1. Đọc dữ liệu từ SQLite

In [2]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM view_pr_clean", conn)
conn.close()
print(f"Kích thước dữ liệu thô: {df.shape}")

Kích thước dữ liệu thô: (1981, 26)


### 2. Xử lý thời gian sống thực tế (duration_minutes) của PR

In [3]:
# Chuyển đổi datetime chuẩn
time_cols = ['created_datetime', 'closed_datetime', 'merged_datetime']
df[time_cols] = df[time_cols].apply(pd.to_datetime)

# Tính toán thời gian sống thực tế của PR theo đơn vị Phút
df['duration_minutes'] = (df['closed_datetime'] - df['created_datetime']).dt.total_seconds() / 60.0
df['duration_hours'] = df['duration_minutes'] / 60.0
df['is_rejected'] = (df['is_merged'] == 0).astype(int)

print(df['duration_minutes'].describe())

count      1981.000000
mean        599.001455
std       20418.586229
min           0.033333
25%           0.166667
50%           0.400000
75%           2.683333
max      907622.683333
Name: duration_minutes, dtype: float64


### 3. Lọc ngoại lệ (Outliers) dựa trên phân phối thực tế (Domain Rules)

In [4]:
# Áp dụng các ngưỡng lọc outliers để loại bỏ PR tự động/rác
df_clean = df[
    (df['additions'] <= 5000) &
    (df['deletions'] <= 5000) &
    (df['changed_files'] <= 50) &
    (df['commits'] <= 50)
].copy()

print(f"Kích thước dữ liệu sạch: {df_clean.shape}")
print(f"Đã loại bỏ: {len(df) - len(df_clean)} dòng ({((len(df) - len(df_clean))/len(df))*100:.2f}%)")

Kích thước dữ liệu sạch: (1771, 27)
Đã loại bỏ: 210 dòng (10.60%)


### 4. Xuất dữ liệu sạch ra file CSV

In [5]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"🎉 Lưu dữ liệu sạch thành công tại: {OUTPUT_PATH}")

🎉 Lưu dữ liệu sạch thành công tại: e:\_FPT_UNI_\Ki 3\ADY\ADY_Final_Project\data\processed\clean_prs.csv
